# Technical Crypto EDA

Notebook dedicado ? carga do OHLCV, cria??o dos indicadores t?cnicos e inspe??o do dataset final.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt

from src.analysis.blockchain_eda import build_missing_report, class_balance, dataset_snapshot
from src.analysis.technical_eda import compare_feature_frames
from src.data.historical_loader import load_historical_asset_dataframe
from src.data.technical_preprocessing import prepare_technical_market_dataframe
from src.features.technical_indicators import (
    DEFAULT_TECHNICAL_HORIZONS,
    add_technical_indicators,
    add_technical_targets,
)


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
HORIZONS = DEFAULT_TECHNICAL_HORIZONS


In [ ]:
raw_asset = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
raw_asset.head()


In [ ]:
technical_base = prepare_technical_market_dataframe(raw_asset)
technical_base.head()


In [ ]:
technical_df = add_technical_indicators(technical_base)
technical_df = add_technical_targets(technical_df, horizons=HORIZONS)
technical_df.head()


In [ ]:
compare_feature_frames(technical_base, technical_df)


In [ ]:
dataset_snapshot(technical_df)


In [ ]:
build_missing_report(technical_df).head(20)


In [ ]:
for horizon in HORIZONS:
    target_column = f'Direction_t+{horizon}'
    print(target_column)
    display(class_balance(technical_df.dropna(subset=[target_column]), target_column))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
axes[0].plot(technical_df.index, technical_df['Close'])
axes[0].set_title('Close')
axes[1].plot(technical_df.index, technical_df['RSI_14'])
axes[1].set_title('RSI_14')
axes[2].plot(technical_df.index, technical_df['OBV'])
axes[2].set_title('OBV')
axes[3].plot(technical_df.index, technical_df['Upper_Band'], label='Upper')
axes[3].plot(technical_df.index, technical_df['Lower_Band'], label='Lower')
axes[3].set_title('Bollinger Bands')
axes[3].legend()
plt.tight_layout()
plt.show()
